### Общее: версия, тип языка, парадигмы

<sub>Слайд 6</sub>

* Версия Python в курсе — **3.9.12**; версия ОС — **Ubuntu 20**.
* Язык **общего назначения** — не предметно-ориентированный.
* **Интерпретируемый**, с **динамической типизацией**.
* Поддерживает парадигмы:
  * объектно-ориентированную;
  * функциональную;
  * процедурную.
* Один из наиболее широко используемых языков в мире.
* Программы на Python исполняются на большинстве распространённых платформ.


**Динамическая типизация: имя — это ярлык**

Проверим на практике, что тип в Python принадлежит значению, а не имени.
Одна и та же переменная последовательно указывает на объекты четырёх разных типов.


In [1]:
label = 42
print(label, '->', type(label).__name__)

label = 'сорок два'
print(label, '->', type(label).__name__)

label = [4, 2]
print(label, '->', type(label).__name__)

label = print
print(label, '->', type(label).__name__)
label('вызов через ярлык: имя функции - тоже просто ссылка')

try:
    print('строка' + 42)
except TypeError as e:
    print('TypeError:', e)

42 -> int
сорок два -> str
[4, 2] -> list
<built-in function print> -> builtin_function_or_method
вызов через ярлык: имя функции - тоже просто ссылка
TypeError: can only concatenate str (not "int") to str


**«Интерпретируемый» — это на самом деле байт-код**

Модуль `dis` показывает, во что превращается наш исходный текст.
Это не машинный код процессора, а инструкции виртуальной машины CPython.


In [2]:
import dis

def add_tax(price):
    return price * 1.2

print('байт-код функции add_tax:')
dis.dis(add_tax)

print()
print('имя функции  :', add_tax.__name__)
print('код-объект   :', add_tax.__code__)
print('константы    :', add_tax.__code__.co_consts)
print('имена аргум. :', add_tax.__code__.co_varnames)

байт-код функции add_tax:
  3           0 RESUME                   0

  4           2 LOAD_FAST                0 (price)
              4 LOAD_CONST               1 (1.2)
              6 BINARY_OP                5 (*)
             10 RETURN_VALUE

имя функции  : add_tax
код-объект   : <code object add_tax at 0x119430fa0, file "/var/folders/z5/mmh4h5s153x_w52pmq7hm3nr0000gn/T/ipykernel_96469/1539868899.py", line 3>
константы    : (None, 1.2)
имена аргум. : ('price',)


### Что можно делать на Python

<sub>Слайд 7</sub>

С помощью Python можно:

* решать задачи системного программирования;
* разрабатывать графические интерфейсы;
* разрабатывать программы для сети Интернет;
* решать **интеграционные** задачи, связывая компоненты, написанные на других ЯП;
* разрабатывать программы для работы с БД;
* быстро **прототипировать**;
* решать численные задачи и задачи научного программирования.


## Архитектура ЭВМ


### Три компонента компьютера

<sub>Слайд 8-9</sub>

Компьютер можно представить в виде трёх компонент:

* **процессор** — выполняет инструкции;
* **память** — хранит данные и сам код;
* **шина** — переносит данные между процессором и памятью.

Классическая схема фон Неймана: код и данные лежат в одной памяти.

Следствие, важное для нас: процессор не работает с памятью напрямую «бесплатно» —
каждое обращение к данным проходит через шину и стоит времени.


**Смотрим на машину, на которой мы работаем**

Заглянем в железо изнутри Python: сколько ядер, какова разрядность,
насколько велики целые числа.


In [3]:
import os, sys, platform

print('логических ядер (os.cpu_count) :', os.cpu_count())
print('архитектура                    :', platform.machine())
print('разрядность указателя, бит     :', sys.maxsize.bit_length() + 1)
print('максимальное «машинное» целое  :', sys.maxsize)
print('размер int 0 в байтах          :', sys.getsizeof(0))
print('размер int 2**100 в байтах     :', sys.getsizeof(2 ** 100))
print('целые в Python не переполняются:', 2 ** 100 + 1)

логических ядер (os.cpu_count) : 14
архитектура                    : arm64
разрядность указателя, бит     : 64
максимальное «машинное» целое  : 9223372036854775807
размер int 0 в байтах          : 28
размер int 2**100 в байтах     : 40
целые в Python не переполняются: 1267650600228229401496703205377


**Сколько стоит объект: sys.getsizeof**

Сравним, сколько памяти занимают разные объекты Python,
и сколько занял бы тот же набор чисел в C-массиве (4 байта на число).


In [4]:
import sys

samples = [0, 1, 10 ** 20, 3.14, True, 'a', 'a' * 100,
           [], [0] * 1000, (), {}, set()]

for obj in samples:
    name = repr(obj)
    if len(name) > 22:
        name = name[:19] + '...'
    print(name.ljust(24), sys.getsizeof(obj), 'байт')

numbers_list = list(range(1000))
python_bytes = sys.getsizeof(numbers_list) + sum(sys.getsizeof(x) for x in numbers_list)
print()
print('список 1000 чисел, только сам список :', sys.getsizeof(numbers_list), 'байт')
print('список 1000 чисел вместе с объектами :', python_bytes, 'байт')
print('те же 1000 чисел как int32 в C       :', 1000 * 4, 'байт')
print('во сколько раз дороже                :', round(python_bytes / 4000, 1))

0                        28 байт
1                        28 байт
100000000000000000000    36 байт
3.14                     24 байт
True                     28 байт
'a'                      50 байт
'aaaaaaaaaaaaaaaaaa...   149 байт
[]                       56 байт
[0, 0, 0, 0, 0, 0, ...   8056 байт
()                       40 байт
{}                       64 байт
set()                    216 байт

список 1000 чисел, только сам список : 8056 байт
список 1000 чисел вместе с объектами : 36056 байт
те же 1000 чисел как int32 в C       : 4000 байт
во сколько раз дороже                : 9.0


### Характеристики процессора: IPC и clock speed

<sub>Слайд 10</sub>

Основные характеристики процессора:

* **IPC** (instructions per cycle) — сколько операций/инструкций он может выполнить
  за один цикл вычислений;
* **clock speed** — сколько циклов он может выполнить за одну секунду.

Процессор с clock speed 4 GHz может выполнить **4 миллиарда** циклов в секунду.

Грубая оценка производительности: `операций в секунду ≈ IPC × clock speed × число ядер`.


### Почему частоты перестали расти и при чём тут GIL

<sub>Слайд 11</sub>

* Создатели процессоров ищут компромисс между IPC и clock speed.
* Рост **clock speed** немедленно ускоряет все программы — больше вычислений в секунду.
* Рост **IPC** тоже ускоряет, в том числе за счёт **векторизации**, если она поддержана.
* В последние годы обе величины растут незначительно: физические ограничения
  на размер транзистора (и тепловыделение).
* Производители перешли к другим методам ускорения:
  * многопоточность;
  * более умное исполнение — **out-of-order execution**;
  * многоядерность.
* В Python есть ограничение на многопоточность — **GIL** (Global Interpreter Lock).


**GIL в действии: два потока не быстрее одного**

Крутим один и тот же счётчик: сначала два раза подряд в одном потоке,
потом те же два раза в двух потоках одновременно. Если бы GIL не было,
второй замер был бы примерно вдвое быстрее. Ячейка считается пару секунд.


In [5]:
import time, threading, os

def spin(limit):
    counter = 0
    while counter < limit:
        counter += 1

WORK = 4000000

start = time.perf_counter()
spin(WORK)
spin(WORK)
serial_time = time.perf_counter() - start

workers = [threading.Thread(target=spin, args=(WORK,)) for _ in range(2)]
start = time.perf_counter()
for w in workers:
    w.start()
for w in workers:
    w.join()
threaded_time = time.perf_counter() - start

print('ядер доступно      :', os.cpu_count())
print('последовательно, с :', round(serial_time, 3))
print('в двух потоках, с  :', round(threaded_time, 3))
print('ускорение          :', round(serial_time / threaded_time, 2), 'раз')

ядер доступно      : 14
последовательно, с : 0.141
в двух потоках, с  : 0.119
ускорение          : 1.19 раз


**А вот где потоки помогают: ожидание**

Тот же опыт, но вместо счёта — ожидание (`time.sleep` имитирует запрос к сети
или чтение с диска). На время ожидания GIL отпускается.


In [6]:
import time, threading

def wait_a_bit(seconds):
    time.sleep(seconds)

PAUSE = 0.3

start = time.perf_counter()
wait_a_bit(PAUSE)
wait_a_bit(PAUSE)
wait_a_bit(PAUSE)
serial_wait = time.perf_counter() - start

waiters = [threading.Thread(target=wait_a_bit, args=(PAUSE,)) for _ in range(3)]
start = time.perf_counter()
for w in waiters:
    w.start()
for w in waiters:
    w.join()
threaded_wait = time.perf_counter() - start

print('три ожидания подряд, с  :', round(serial_wait, 3))
print('три ожидания в потоках,с:', round(threaded_wait, 3))
print('ускорение               :', round(serial_wait / threaded_wait, 2), 'раз')

три ожидания подряд, с  : 0.912
три ожидания в потоках,с: 0.306
ускорение               : 2.98 раз


### Память: последовательное чтение, произвольный доступ, задержка

<sub>Слайд 12</sub>

* Основная разница между типами памяти — **скорость чтения/записи**.
* Скорость зависит от того, **как** данные считываются: большинство носителей
  эффективнее, когда читают один большой блок, а не много маленьких.
  Это **sequential read** против **random access**.
* Аналогия: проще читать книгу страница за страницей, чем открывать страницы
  в случайном порядке.
* Способ чтения важен для всех типов носителей, но степень важности разная.
* Вторая ключевая характеристика — **задержка (latency)**: время, которое нужно
  носителю, чтобы *найти* нужные данные.
* У жёсткого диска (spinning hard drive) задержка велика: надо раскрутить диск
  и переставить головку. У RAM такой проблемы нет.


**Последовательный обход против случайного**

Один и тот же список, одна и та же сумма, одинаковое число операций.
Разница только в порядке обращения к элементам. Ячейка считается несколько секунд.


In [7]:
import time, random

SIZE = 2000000
big_array = list(range(SIZE))

seq_index = list(range(SIZE))
rnd_index = list(range(SIZE))
random.seed(42)
random.shuffle(rnd_index)

start = time.perf_counter()
total_seq = 0
for i in seq_index:
    total_seq += big_array[i]
time_seq = time.perf_counter() - start

start = time.perf_counter()
total_rnd = 0
for i in rnd_index:
    total_rnd += big_array[i]
time_rnd = time.perf_counter() - start

print('суммы совпали      :', total_seq == total_rnd)
print('по порядку, с      :', round(time_seq, 3))
print('вразнобой, с       :', round(time_rnd, 3))
print('во сколько дороже  :', round(time_rnd / time_seq, 2), 'раз')

суммы совпали      : True
по порядку, с      : 0.082
вразнобой, с       : 0.532
во сколько дороже  : 6.48 раз


### Иерархия памяти

<sub>Слайд 13-14</sub>

Сверху вниз — медленнее и больше, снизу вверх — быстрее и дороже:

| Уровень | Типичный объём | Типичная задержка |
|---|---|---|
| 1. Дисковые накопители | терабайты | 0.1–10 мс |
| 2. Оперативная память (RAM) | десятки ГБ | ~100 нс |
| 3. Кэш процессора (L3 / L2 / L1) | МБ / сотни КБ / десятки КБ | ~40 / ~10 / ~1 нс |
| 4. Регистры | сотни байт | доли такта |

* Между соседними уровнями разрыв в скорости — примерно на порядок.
* Между диском и регистрами — **шесть-семь порядков**.
* Данные всегда «поднимаются» снизу вверх блоками (кэш-линия ≈ 64 байта).


**Иерархия и структура данных: insert(0) против deque**

Список хранит элементы подряд, поэтому вставка в начало сдвигает всю память.
`collections.deque` устроен как связанные блоки — у него вставка в начало дешёвая.


In [8]:
import time, collections

COUNT = 50000

plain_list = []
start = time.perf_counter()
for i in range(COUNT):
    plain_list.insert(0, i)
time_list = time.perf_counter() - start

fast_deque = collections.deque()
start = time.perf_counter()
for i in range(COUNT):
    fast_deque.appendleft(i)
time_deque = time.perf_counter() - start

print('одинаковый результат :', list(fast_deque)[:5] == plain_list[:5])
print('list.insert(0, x), с :', round(time_list, 4))
print('deque.appendleft, с  :', round(time_deque, 4))
print('во сколько раз хуже  :', round(time_list / max(time_deque, 1e-9)))

одинаковый результат : True
list.insert(0, x), с : 0.1496
deque.appendleft, с  : 0.0015
во сколько раз хуже  : 99


**Локальность: обход матрицы по строкам и по столбцам**

Матрица 900×900 как список списков. Суммируем все элементы двумя способами:
идя по строкам (как данные лежат) и по столбцам (поперёк).


In [9]:
import time

N = 900
matrix = [[r * N + c for c in range(N)] for r in range(N)]

start = time.perf_counter()
total_rows = 0
for row in matrix:
    for value in row:
        total_rows += value
time_rows = time.perf_counter() - start

start = time.perf_counter()
total_cols = 0
for c in range(N):
    for r in range(N):
        total_cols += matrix[r][c]
time_cols = time.perf_counter() - start

print('суммы совпали   :', total_rows == total_cols)
print('по строкам, с   :', round(time_rows, 3))
print('по столбцам, с  :', round(time_cols, 3))
print('отношение       :', round(time_cols / time_rows, 2))

суммы совпали   : True
по строкам, с   : 0.027
по столбцам, с  : 0.055
отношение       : 2.07


**perf_counter против time: зачем два таймера**

Мы всё время пользуемся `time.perf_counter`. Посмотрим, чем он отличается
от привычного `time.time` и какое у обоих разрешение.


In [10]:
import time

print('time.time()        :', time.time())
print('perf_counter()     :', time.perf_counter())
print()
print('разрешение time         , с:', time.get_clock_info('time').resolution)
print('разрешение perf_counter , с:', time.get_clock_info('perf_counter').resolution)
print('perf_counter монотонный   :', time.get_clock_info('perf_counter').monotonic)
print('time монотонный           :', time.get_clock_info('time').monotonic)
print()
samples = []
for _ in range(5):
    a = time.perf_counter()
    b = time.perf_counter()
    samples.append(b - a)
print('два вызова подряд, с:', [format(s, '.9f') for s in samples])
print('минимальный шаг ~   :', format(min(s for s in samples if s > 0), '.9f'), 'с')

time.time()        : 1789546904.56566
perf_counter()     : 1065079.508230125

разрешение time         , с: 1.0000000000000002e-06
разрешение perf_counter , с: 4.166666666666667e-08
perf_counter монотонный   : True
time монотонный           : False

два вызова подряд, с: ['0.000000708', '0.000000084', '0.000000084', '0.000000083', '0.000000083']
минимальный шаг ~   : 0.000000083 с


## Шина


### Характеристики шины: ширина и частота

<sub>Слайд 15</sub>

Основные характеристики шины:

* **bus width** — сколько данных может быть передано за один сеанс передачи;
* **bus frequency** — сколько сеансов передачи может быть выполнено в секунду.

Важно отметить:

* данные, передаваемые за один сеанс, всегда **последовательные**: часть данных
  читается из памяти и перемещается в другое место;
* возможность передавать больше данных за сеанс позволяет процессорам
  с **векторными инструкциями** работать эффективнее;
* при небольшой ширине и высокой частоте мы хорошо обрабатываем данные,
  которые нужно **часто** считывать из RAM.

Грубо: `пропускная способность ≈ bus width × bus frequency`.


### Почему шина решает в машинном обучении

<sub>Слайд 16</sub>

Три следствия, которые вы будете видеть весь курс:

* **Векторизация имеет смысл только на плотных данных.** Список указателей
  векторному блоку не отдать — нужны числа подряд, как в `array.array` или NumPy.
* **Многие операции ограничены памятью, а не арифметикой** (memory-bound):
  поэлементное сложение двух больших массивов, нормализация, функции активации.
  Считать там почти нечего, всё время уходит на перевозку данных.
* **Передача между устройствами — отдельная шина и отдельная плата.**
  CPU ↔ GPU идёт по PCIe, и она на порядок медленнее памяти самой видеокарты.

Практические правила: держать данные плотно и в нужном типе (`float32` вместо
`float64` — вдвое меньше перевозить), не гонять тензоры туда-сюда внутри цикла
обучения, объединять мелкие операции в одну (fusion).


**Перевозка данных: плотный массив против списка объектов**

`array.array` из стандартной библиотеки хранит числа плотно, как в C.
Сравним расход памяти и время поэлементного суммирования.


In [11]:
import sys, time, array

COUNT = 1000000
py_list = list(range(COUNT))
dense = array.array('i', range(COUNT))

list_bytes = sys.getsizeof(py_list) + sum(sys.getsizeof(x) for x in py_list[:1000]) * (COUNT // 1000)
print('список, оценка памяти, МБ :', round(list_bytes / 1048576, 1))
print('array(int32), памяти, МБ  :', round(dense.buffer_info()[1] * dense.itemsize / 1048576, 1))

start = time.perf_counter()
sum_list = sum(py_list)
t_list = time.perf_counter() - start

start = time.perf_counter()
sum_dense = sum(dense)
t_dense = time.perf_counter() - start

print('суммы равны               :', sum_list == sum_dense)
print('sum по списку, с          :', round(t_list, 4))
print('sum по array, с           :', round(t_dense, 4))

список, оценка памяти, МБ : 34.3
array(int32), памяти, МБ  : 3.8
суммы равны               : True
sum по списку, с          : 0.0028
sum по array, с           : 0.0054


**Сколько байт в секунду мы вообще двигаем**

Грубая оценка пропускной способности, доступной из Python:
копируем большой блок байт и считаем мегабайты в секунду.


In [12]:
import time

BLOCK = b'x' * (8 * 1024 * 1024)
REPEATS = 20

start = time.perf_counter()
for _ in range(REPEATS):
    copy_of_block = BLOCK[:]
elapsed = time.perf_counter() - start

moved_mb = len(BLOCK) * REPEATS / 1048576
print('размер блока, МБ    :', len(BLOCK) // 1048576)
print('перевезено всего, МБ:', round(moved_mb, 1))
print('время, с            :', round(elapsed, 4))
print('скорость, МБ/с      :', round(moved_mb / elapsed))
print('копия равна оригиналу:', copy_of_block == BLOCK)

размер блока, МБ    : 8
перевезено всего, МБ: 160.0
время, с            : 0.0
скорость, МБ/с      : 7573980
копия равна оригиналу: True


### Итоги блока: что унести с собой

<sub>Слайд 8-16</sub>

**Процессор**

* `IPC × clock speed × ядра` — пиковая, недостижимая на практике оценка.
* Частоты не растут с середины 2000-х: физика транзистора и тепло.
* Ускорение теперь берут из многоядерности, out-of-order и векторизации.
* **GIL**: байт-код исполняет один поток. Потоки помогают на ожидании, не на счёте.
  Для счёта — `multiprocessing` или библиотека на C.

**Память**

* Иерархия: регистры → кэш → RAM → диск, разрыв между краями ~10⁶.
* Последовательный доступ дешевле произвольного на **любом** носителе.
* Объект Python дорог: `int` ≈ 28 байт против 4 байт в C.

**Шина**

* `bus width × bus frequency`; перевозит только непрерывные куски.
* Многие операции в МО — memory-bound, а не compute-bound.

**Рабочее правило курса:** данные храним плотно, цикл выносим на уровень C.


## Как исполняется программа на Python


### Программа на Python: из чего она состоит

<sub>Слайд 17</sub>

* Python — **интерпретируемый** язык: для запуска программы нужен установленный интерпретатор.
* По соглашению файлы имеют расширение `.py`. Формально это обязательно только для файлов, которые **импортируются**.
* Структура программы — четыре уровня:
  1. программы состоят из **модулей**;
  2. модули содержат **операторы** (statements);
  3. операторы содержат **выражения** (expressions);
  4. выражения создают и обрабатывают **объекты**.
* Модуль — это один файл `.py`. Пакет — каталог с модулями.


### Конец строки, отступы, байт-код

<sub>Слайд 18-19</sub>

* **Конец строки завершает оператор** — никаких точек с запятой.
* Вложенные операторы объединяются в **блок** по физическим отступам — никаких фигурных скобок.
* Запуск программы = выполнение всех операторов файла с начала до конца, друг за другом.
* Сначала исходный код **компилируется в байт-код**, затем виртуальная машина его **исполняет**.
* Байт-код — низкоуровневое платформенно-независимое представление: каждый оператор превращается в набор инструкций. Байт-код выполняется быстрее исходного текста.


**Смотрим на настоящий байт-код: модуль dis**

Слово «байт-код» звучит абстрактно, поэтому давайте его увидим.
Модуль `dis` (disassembler) показывает инструкции, в которые превратилась функция.


In [13]:
import dis

def tax_total(price, rate):
    total = price * (1 + rate)
    return round(total, 2)

print('результат функции:', tax_total(100, 0.2))
print('-' * 40)
dis.dis(tax_total)

результат функции: 120.0
----------------------------------------
  3           0 RESUME                   0

  4           2 LOAD_FAST                0 (price)
              4 LOAD_CONST               1 (1)
              6 LOAD_FAST                1 (rate)
              8 BINARY_OP                0 (+)
             12 BINARY_OP                5 (*)
             16 STORE_FAST               2 (total)

  5          18 LOAD_GLOBAL              1 (NULL + round)
             30 LOAD_FAST                2 (total)
             32 LOAD_CONST               2 (2)
             34 PRECALL                  2
             38 CALL                     2
             48 RETURN_VALUE


**compile(): объект кода можно подержать в руках**

Компиляцию можно вызвать вручную — встроенной функцией `compile()`.
Она возвращает объект кода, у которого есть интересные атрибуты.


In [14]:
source_text = '''greeting = 'привет'
shout = greeting.upper()
print(shout, 42)'''

code_obj = compile(source_text, '<лекция>', 'exec')
print('тип объекта   :', type(code_obj))
print('co_consts     :', code_obj.co_consts)
print('co_names      :', code_obj.co_names)
print('имя файла     :', code_obj.co_filename)
print('-' * 40)
exec(code_obj)

тип объекта   : <class 'code'>
co_consts     : ('привет', 42, None)
co_names      : ('greeting', 'upper', 'shout', 'print')
имя файла     : <лекция>
----------------------------------------
ПРИВЕТ 42


### .pyc, __pycache__ и общая схема исполнения

<sub>Слайд 20-21</sub>

* Скомпилированный байт-код сохраняется в файл `.pyc` в подкаталоге `__pycache__`.
* Это оптимизация **последующих запусков**: если исходник не менялся и версия Python та же — этап компиляции пропускается.
* Байт-код сохраняется **только для импортируемых** файлов, но не для файла верхнего уровня, запущенного как сценарий.
* Имя файла содержит версию: `greeter.cpython-311.pyc` — байт-код разных версий не смешивается.
* Общая схема: `исходный код .py` → **компилятор** → `байт-код .pyc` → **виртуальная машина (PVM)** → результат.


**Создаём свой модуль и импортируем его**

Сделаем файл-модуль в каталоге `DEMO`, добавим этот каталог в пути поиска
и импортируем. Важно увидеть: импорт **выполняет** код модуля.


In [15]:
import sys

module_src = '''print('>>> модуль lecture_greeter выполняется!')
TITLE = 'приветствие, версия 1'

def greet(name):
    return 'Привет, ' + name + '!'
'''

(DEMO / 'lecture_greeter.py').write_text(module_src, encoding='utf-8')
sys.path.insert(0, str(DEMO.resolve()))

import lecture_greeter
print('вернула функция:', lecture_greeter.greet('ВМК'))
print('атрибут TITLE  :', lecture_greeter.TITLE)

NameError: name 'DEMO' is not defined

**Где же обещанный .pyc**

Раз модуль был импортирован, рядом с ним должен появиться `__pycache__`.
Проверяем.


In [ ]:
cache_dir = DEMO / '__pycache__'
print('каталог __pycache__ существует:', cache_dir.exists())

if cache_dir.exists():
    for pyc_file in sorted(cache_dir.iterdir()):
        print('  ', pyc_file.name, '-', pyc_file.stat().st_size, 'байт')

py_size = (DEMO / 'lecture_greeter.py').stat().st_size
print('размер исходника  :', py_size, 'байт')
print('первые 16 байт pyc:', next(cache_dir.iterdir()).read_bytes()[:16])

каталог __pycache__ существует: True
   lecture_greeter.cpython-311.pyc - 479 байт
   lecture_main_demo.cpython-311.pyc - 572 байт
   lecture_runner.cpython-311.pyc - 432 байт
размер исходника  : 174 байт
первые 16 байт pyc: b'\xa7\r\r\n\x00\x00\x00\x00.P\xaaj\xd3\x00\x00\x00'


## Реализации языка и виртуальные окружения


### Существующие реализации языка Python

<sub>Слайд 22</sub>

* **CPython** — стандартная (эталонная) реализация, написана на C. Быстрее, полнее, современнее и надёжнее альтернатив. Когда говорят «Python», имеют в виду её.
* **Jython** — реализация на Java: компилирует исходный код Python в байт-код Java и отдаёт его JVM. Позволяет смешивать классы Java и Python.
* **IronPython** — реализация на C# для платформы .NET, очень похожа на Jython, автор тот же.
* **PyPy** — реализация, ориентированная на производительность: **JIT-компилятор** (just-in-time, компиляция «на лету») и поддержка микропотоков для массового параллелизма.
* **Cython** — компилирует код Python в код C и даёт полностью компилируемую программу; требует добавления информации о типах.
* Сегодня к списку добавились **MicroPython** (микроконтроллеры), **Pyodide** (Python в браузере через WebAssembly), **GraalPy**, а также **Numba** — JIT для числового кода поверх обычного CPython.


**Какая именно реализация исполняет наш ноутбук**

Спросим у самого интерпретатора, кто он такой. Это первое, что стоит
выяснять, когда чужой код у вас не работает, а у автора работает.


In [ ]:
import platform, sys

print('реализация      :', platform.python_implementation())
print('версия          :', platform.python_version())
print('версия кортежем :', sys.version_info[:3])
print('имя реализации  :', sys.implementation.name)
print('тег байт-кода   :', sys.implementation.cache_tag)
print('платформа       :', sys.platform)
print('компилятор сборки:', platform.python_compiler())

реализация      : CPython
версия          : 3.11.11
версия кортежем : (3, 11, 11)
имя реализации  : cpython
тег байт-кода   : cpython-311
платформа       : darwin
компилятор сборки: Clang 18.1.8 


### Виртуальные окружения: venv

<sub>Слайд 23</sub>

* Программы почти всегда используют пакеты **вне стандартной библиотеки**.
* Разным программам нужны **разные версии** одних и тех же пакетов; различаться могут и версии самого интерпретатора.
* Чтобы не возникало коллизий и всё это уживалось на одной машине, придуман механизм **виртуальных окружений** — модуль `venv` стандартной библиотеки.

```bash
# 1. создать окружение
python3 -m venv tutorial-env

# 2. активировать
source tutorial-env/bin/activate       # Linux / macOS
tutorial-env\Scripts\activate.bat      # Windows

# 3. деактивировать
deactivate
```

* Окружение — это каталог с копией (или ссылкой на) интерпретатора и **собственным** `site-packages`.
* Каталог окружения **не кладут** в git: в репозиторий кладут `requirements.txt`.


**Мы внутри виртуального окружения или нет?**

Активность окружения — не магия, её видно из самого Python.
Ключ — пара `sys.prefix` и `sys.base_prefix`.


In [ ]:
import sys, os

print('sys.executable :', sys.executable)
print('sys.prefix     :', sys.prefix)
print('sys.base_prefix:', sys.base_prefix)
print('окружение активно:', sys.prefix != sys.base_prefix)
print('VIRTUAL_ENV    :', repr(os.environ.get('VIRTUAL_ENV', '')))

site_dirs = [p for p in sys.path if 'site-packages' in p or 'dist-packages' in p]
print('каталоги пакетов:')
for path_entry in site_dirs[:3]:
    print('   ', path_entry)

sys.executable : /Users/air/miniconda3/bin/python
sys.prefix     : /Users/air/miniconda3
sys.base_prefix: /Users/air/miniconda3
окружение активно: False
VIRTUAL_ENV    : ''
каталоги пакетов:
    /Users/air/miniconda3/lib/python3.11/site-packages


### Виртуальные окружения: conda

<sub>Слайд 24</sub>

```bash
# создать окружение, задав версию Python и сразу поставив пакеты
conda create --name env_name python==3.7.5 package_name1 package_name2

# активировать / деактивировать
conda activate env_name
conda deactivate

# удалить окружение
conda env remove -n env_name

# установить пакеты
conda install pkg_name1==1.x.y pkg_name2==1.x.y
conda install --file requirements.txt
conda install jupyter -n tdc-starter-kit
pip install pkg_name1==1.x.y pkg_name2==1.x.y
pip install -r requirements.txt

# список существующих окружений
conda env list        # или: conda info --envs
```

* В отличие от `venv`, conda умеет ставить **сам Python нужной версии** и не-Python зависимости (компиляторы, CUDA, BLAS).
* Окружения живут в одном месте и адресуются **по имени**, а не по пути.


## Способы запуска программы


### Шесть способов запуска. Командная строка

<sub>Слайд 25-26</sub>

Способы запуска программы на Python:
1. **Командная строка**;
2. **интерактивная среда** (REPL, IPython, Jupyter);
3. **импорт**;
4. **IDE**;
5. **встраивание** в программу на другом языке, например на C;
6. **фиксированный двоичный файл** с байт-кодом и виртуальной машиной внутри.

```bash
# передать путь к файлу с программой
echo "print('hello world')" > script.py
python script.py

# передать саму программу параметром
python -c "print('hello world')"

# запустить как исполняемый сценарий
echo "#!/usr/bin/env python3" > script.py
echo "print('hello world')" >> script.py
chmod +x script.py
./script.py
```


**Запускаем файл из командной строки прямо отсюда**

Создадим настоящий файл-сценарий в каталоге `DEMO` и запустим его отдельным
процессом — так же, как это сделала бы командная строка.


In [ ]:
import os, sys

script_path = (DEMO / 'lecture_script.py').resolve()
script_path.write_text(
    "print('привет из отдельного процесса')\n"
    "import sys\n"
    "print('argv:', sys.argv)\n", encoding='utf-8')

py = '"' + sys.executable + '"'
print('--- python script.py ---')
print(os.popen(py + ' "' + str(script_path) + '" alpha 7').read().rstrip())
print('--- python -c ... ---')
print(os.popen(py + " -c 'print(sum(range(101)))'").read().rstrip())

--- python script.py ---
привет из отдельного процесса
argv: ['/Users/air/Downloads/demo/lecture_script.py', 'alpha', '7']
--- python -c ... ---
5050


**Shebang и chmod +x**

Теперь третий вариант со слайда: сделаем файл самостоятельной программой,
которую запускают по имени, без слова `python`.


In [ ]:
import os, stat

exe_path = (DEMO / 'lecture_exe.py').resolve()
exe_path.write_text(
    "#!/usr/bin/env python3\n"
    "print('я знаю, каким интерпретатором меня запускать')\n", encoding='utf-8')

print('права до chmod :', oct(exe_path.stat().st_mode)[-3:])
exe_path.chmod(exe_path.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP)
print('права после    :', oct(exe_path.stat().st_mode)[-3:])
print('первая строка  :', exe_path.read_text(encoding='utf-8').splitlines()[0])
print('запуск по имени:', os.popen('"' + str(exe_path) + '"').read().rstrip())

права до chmod : 754
права после    : 754
первая строка  : #!/usr/bin/env python3
запуск по имени: я знаю, каким интерпретатором меня запускать


### Импорт как способ запуска. reload

<sub>Слайд 27</sub>

* Большие программы делят на независимые функциональные блоки в отдельных файлах — **модулях**; это даёт переиспользование внутри проекта и между проектами.
* Один модуль обычно **главный**: он запускается первым и импортирует остальные.
* Операция импорта **выполняет код** загружаемого файла — значит, импорт тоже является способом запуска:

```bash
echo "print('hello world')" > script.py
python -c "import script"
hello world
```

* Но срабатывает это **только один раз за сеанс**: повторный `import` ничего не выполняет.
* Чтобы выполнить файл заново, не перезапуская интерпретатор, нужна функция `reload` из модуля `importlib`.


**Повторный импорт молчит**

Создадим модуль, который громко объявляет о своём выполнении,
и импортируем его дважды подряд.


In [ ]:
import sys

runner_src = (
    "print('>>> lecture_runner: код модуля выполнен')\n"
    "VERSION = 1\n"
    "LIMIT = 100\n")
(DEMO / 'lecture_runner.py').write_text(runner_src, encoding='utf-8')
sys.path.insert(0, str(DEMO.resolve()))

print('первый импорт:')
import lecture_runner
print('второй импорт:')
import lecture_runner
print('(между ними ничего не напечаталось)')
print('модуль в sys.modules:', 'lecture_runner' in sys.modules)
print('тот же объект       :', sys.modules['lecture_runner'] is lecture_runner)

первый импорт:
>>> lecture_runner: код модуля выполнен
второй импорт:
(между ними ничего не напечаталось)
модуль в sys.modules: True
тот же объект       : True


**importlib.reload выполняет файл заново**

Изменим файл модуля на диске и посмотрим, что видит программа —
до перезагрузки и после.


In [ ]:
import importlib

new_src = (
    "print('>>> lecture_runner: выполнен снова, это уже вторая редакция файла')\n"
    "VERSION = 2\n"
    "LIMIT = 100\n"
    "NOTE = 'имя, которого в первой редакции не было'\n")
(DEMO / 'lecture_runner.py').write_text(new_src, encoding='utf-8')

print('до reload   : VERSION =', lecture_runner.VERSION,
      '| есть NOTE:', hasattr(lecture_runner, 'NOTE'))
importlib.reload(lecture_runner)
print('после reload: VERSION =', lecture_runner.VERSION,
      '| есть NOTE:', hasattr(lecture_runner, 'NOTE'))

до reload   : VERSION = 1 | есть NOTE: False
>>> lecture_runner: выполнен снова, это уже вторая редакция файла
после reload: VERSION = 2 | есть NOTE: True


**Имя __main__: один файл, два способа запуска**

Как файл понимает, запустили его напрямую или импортировали?
По значению переменной `__name__`.


In [ ]:
import os, sys

main_demo = (DEMO / 'lecture_main_demo.py').resolve()
main_demo.write_text(
    "print('внутри файла __name__ =', __name__)\n"
    "def area(side):\n"
    "    return side * side\n"
    "if __name__ == '__main__':\n"
    "    print('меня запустили как программу, считаю:', area(5))\n",
    encoding='utf-8')

print('--- запуск как сценария ---')
print(os.popen('"' + sys.executable + '" "' + str(main_demo) + '"').read().rstrip())
print('--- импорт того же файла ---')
import lecture_main_demo
print('area(5) из импортированного модуля:', lecture_main_demo.area(5))

--- запуск как сценария ---
внутри файла __name__ = __main__
меня запустили как программу, считаю: 25
--- импорт того же файла ---
внутри файла __name__ = lecture_main_demo
area(5) из импортированного модуля: 25


### Модуль как пространство имён. dir() и sys.path

<sub>Слайд 28</sub>

* Модуль — это **пакет имён переменных**, то есть **пространство имён**; имена внутри него называются **атрибутами**.
* Импортирующая сторона получает доступ ко всем именам, определённым **на верхнем уровне** файла модуля: функциям, классам, переменным.
* Извлечь имена снаружи можно операторами `import` и `from`, а также вызовом `reload`.
* Все атрибуты модуля перечисляет функция `dir(module)`.
* Имена с **двумя подчёркиваниями** по краям — служебные, их создаёт и использует интерпретатор: `__name__`, `__file__`, `__doc__`, `__spec__`.
* Импортируемые модули Python ищет в каталогах из списка `sys.path`, который инициализируется переменной окружения **PYTHONPATH** плюс набором стандартных каталогов.


**Заглядываем внутрь модуля: dir и dunder-имена**

Посмотрим на наш модуль как на пространство имён: сначала всё подряд,
потом отдельно служебное и отдельно полезное.


In [ ]:
print('__name__:', lecture_runner.__name__)
print('__file__:', lecture_runner.__file__)
print('__doc__ :', repr(lecture_runner.__doc__))

all_names = dir(lecture_runner)
print('всего имён:', len(all_names))

dunder = [name for name in all_names if name.startswith('__')]
public = [name for name in all_names if not name.startswith('_')]
print('служебные :', dunder)
print('полезные  :', public)
print('словарь модуля напрямую:', [k for k in vars(lecture_runner) if not k.startswith('_')])

__name__: lecture_runner
__file__: /Users/air/Downloads/demo/lecture_runner.py
__doc__ : None
всего имён: 11
служебные : ['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']
полезные  : ['LIMIT', 'NOTE', 'VERSION']
словарь модуля напрямую: ['VERSION', 'LIMIT', 'NOTE']


**sys.path: где Python ищет модули**

Посмотрим на список путей поиска и на переменную окружения `PYTHONPATH`,
из которой он частично собирается.


In [ ]:
import sys, os

print('PYTHONPATH:', repr(os.environ.get('PYTHONPATH', '(не задана)')))
print('первые элементы sys.path:')
for i, path_entry in enumerate(sys.path[:5]):
    print('  ', i, repr(path_entry))

my_dir = str((DEMO / 'extra_modules').resolve())
os.makedirs(my_dir, exist_ok=True)
if my_dir not in sys.path:
    sys.path.append(my_dir)
print('свой каталог добавлен, он теперь последний:', sys.path[-1] == my_dir)
print('всего каталогов в поиске:', len(sys.path))

PYTHONPATH: '(не задана)'
первые элементы sys.path:
   0 '/Users/air/Downloads/demo'
   1 '/Users/air/Downloads/demo'
   2 '/Users/air/miniconda3/lib/python311.zip'
   3 '/Users/air/miniconda3/lib/python3.11'
   4 '/Users/air/miniconda3/lib/python3.11/lib-dynload'
свой каталог добавлен, он теперь последний: True
всего каталогов в поиске: 8


### Способ запуска: exec(open(...).read())

<sub>Слайд 29</sub>

* Вызов `exec(open('module1.py').read())` — ещё один приём запуска файла из интерактивной среды, **без импортирования** и последующей перезагрузки.
* Каждый такой вызов выполняет **текущую** версию кода, прочитанную из файла: перезагрузка не нужна.
* Но: `exec` работает так, **как если бы вы поместили код на его место** — подобно оператору `from`.
* Следствие: он способен **молча переписать** переменные, которые вы используете прямо сейчас.
* Никакого пространства имён не создаётся: имена из файла попадают прямо в ваше.
* Отдельная беда — `exec` от данных, полученных извне: это прямое выполнение чужого кода.


**Как exec молча переписывает ваши переменные**

Соберём типичную ситуацию: у нас в работе есть переменные,
и мы «подгружаем настройки» из файла вызовом `exec`. Смотрим на последствия.


In [ ]:
model_name = 'ResNet-50'
threshold = 0.9
accuracy = 0.97

config_path = (DEMO / 'lecture_config.py').resolve()
config_path.write_text("model_name = 'случайный лес'\nthreshold = 0.5\n", encoding='utf-8')

print('до  exec: model_name =', model_name, '| threshold =', threshold)
exec(open(str(config_path), encoding='utf-8').read())
print('после   : model_name =', model_name, '| threshold =', threshold)
print('accuracy уцелела:', accuracy)

до  exec: model_name = ResNet-50 | threshold = 0.9
после   : model_name = случайный лес | threshold = 0.5
accuracy уцелела: 0.97


### Способ запуска: IDE

<sub>Слайд 30</sub>

* **PyCharm** — среда, специализированная под Python: индексирует проект целиком, знает про виртуальные окружения, умеет рефакторинг.
* **VS Code** — универсальный редактор с расширением для Python; сегодня самый распространённый вариант.
* **Eclipse** (с плагином PyDev) — исторический вариант, в новых проектах встречается редко.
* Что даёт IDE поверх «запустить файл»: **отладчик** с точками останова и пошаговым выполнением, автодополнение, переход к определению, проверка стиля.
* Отдельная ветка — **Jupyter Notebook / JupyterLab**: интерактивная среда, в которой мы сейчас находимся.
* Для отладки без IDE есть встроенный модуль `pdb`; `breakpoint()` останавливает программу в нужном месте.


### Встраивание в программу на C и фиксированные двоичные файлы

<sub>Слайд 31-32</sub>

**Встраивание (embedding).** Программу на Python можно вызвать из программы на другом языке:

```c
#include <Python.h>
Py_Initialize();
PyRun_SimpleString("print('Hello world')");
```

* Актуально для программ, которые компилируются и после релиза у пользователя не меняются: Python делает их гибкими.
* Пользователь может **менять поведение системы**, подставляя или правя код на Python, — перекомпиляция всей системы не нужна.
* Включающая система может быть написана на C, C++ и даже на Java (при использовании Jython).

**Фиксированные двоичные файлы (frozen binaries).**

* Пакет, объединяющий **байт-код программы и сам интерпретатор** в одну исполняемую программу.
* Это **не** выход настоящего компилятора: байт-код всё равно исполняется виртуальной машиной, поэтому скорость та же, что у исходных файлов (может выиграть только время старта).
* Python не требуется устанавливать на стороне получателя.
* Код лучше скрыт от пользователя — но «лучше» не значит «надёжно».
* Инструменты: PyInstaller, cx_Freeze, py2exe, Nuitka.


## Лексическая структура


### Из чего состоит текст программы. Комментарии

<sub>Слайд 33-34</sub>

Лексическая структура Python состоит из:
1. **комментариев**;
2. **идентификаторов**;
3. **ключевых слов**;
4. **литералов**;
5. **операторов**;
6. **разделителей**.

**Комментарии:**

* однострочный — от символа `#` до конца строки: `# It's a comment`;
* «многострочный» — строка в тройных кавычках:

```python
"""
It's a comment
Several rows
"""
```

* Строго говоря, второе — не комментарий, а **строковый литерал**, который вычисляется и выбрасывается; в начале модуля, функции или класса он становится **docstring** и попадает в `__doc__`.


**Docstring против комментария**

Сравним три вещи: комментарий, строку-«комментарий» в середине тела
и настоящую строку документации.


In [ ]:
import dis

def area_documented(side):
    """Площадь квадрата со стороной side."""
    return side * side

def area_commented(side):
    side = abs(side)
    'эта строка просто вычислится и будет выброшена'
    # а это комментарий, его в байт-коде нет вообще
    return side * side

print('__doc__ первой :', repr(area_documented.__doc__))
print('__doc__ второй :', repr(area_commented.__doc__))
print('константы первой:', area_documented.__code__.co_consts)
print('константы второй:', area_commented.__code__.co_consts)
print('docstring метода str.upper:', str.upper.__doc__)

__doc__ первой : 'Площадь квадрата со стороной side.'
__doc__ второй : None
константы первой: ('Площадь квадрата со стороной side.',)
константы второй: (None,)
docstring метода str.upper: Return a copy of the string converted to uppercase.


### Идентификаторы и правила именования

<sub>Слайд 35</sub>

* Могут содержать буквенно-цифровые знаки и подчёркивание.
* Могут начинаться с подчёркивания.
* **Не могут** начинаться с цифры.
* **Чувствительны к регистру**: `data`, `Data` и `DATA` — три разных имени.
* Не могут совпадать с ключевым словом.
* **Змеиная нотация** (snake_case) — для переменных и функций: `train_size`, `load_dataset`.
* **Верблюжья нотация** (CamelCase) — для классов: `RandomForest`, `DataLoader`.
* Дополнительно из PEP 8: `КОНСТАНТЫ_ЗАГЛАВНЫМИ`, `_внутреннее` — с одним подчёркиванием, `__служебное__` — не придумываем сами.


**Проверяем правила именования**

У строк есть метод `isidentifier`, который отвечает,
годится ли строка на роль имени. Проверим им все спорные случаи.


In [ ]:
candidates = ['data', '_data', 'data_2', '2data', 'my data',
              'my-data', 'класс_модели', 'класс модели', 'lambda', 'Data']
for text in candidates:
    print(repr(text).ljust(16), 'идентификатор:', text.isidentifier())

Data = 'с большой буквы'
data = 'с маленькой'
print('разные имена:', Data, '|', data, '| одно и то же:', Data is data)

число_объектов = 3
print('кириллица работает:', число_объектов * 2)

'data'           идентификатор: True
'_data'          идентификатор: True
'data_2'         идентификатор: True
'2data'          идентификатор: False
'my data'        идентификатор: False
'my-data'        идентификатор: False
'класс_модели'   идентификатор: True
'класс модели'   идентификатор: False
'lambda'         идентификатор: True
'Data'           идентификатор: True
разные имена: с большой буквы | с маленькой | одно и то же: False
кириллица работает: 6


### Ключевые слова

<sub>Слайд 36</sub>

* Ключевые (зарезервированные) слова **не могут** использоваться как обычные идентификаторы и пишутся **точно** так, как указано ниже.
* В Python 3.9 их 35:

```
False      await      else       import     pass
None       break      except     in         raise
True       class      finally    is         return
and        continue   for        lambda     try
as         def        from       nonlocal   while
assert     del        global     not        with
async      elif       if         or         yield
```

* Актуальный список всегда доступен как `keyword.kwlist`, проверка — `keyword.iskeyword(s)`.
* Отдельно есть **мягкие** ключевые слова (`keyword.softkwlist`): `match`, `case`, `_` — они ключевые только в определённом контексте и именами быть могут.
* Попытка использовать ключевое слово как имя — это `SyntaxError`, то есть ошибка **времени компиляции**.


**keyword: список, проверка и SyntaxError**

Попросим список ключевых слов у самого Python, а потом
намеренно попробуем использовать одно из них как имя.


In [ ]:
import keyword

print('всего ключевых слов:', len(keyword.kwlist))
print('первые десять :', keyword.kwlist[:10])
print('мягкие        :', getattr(keyword, 'softkwlist', []))
print('iskeyword("for")   :', keyword.iskeyword('for'))
print('iskeyword("forest"):', keyword.iskeyword('forest'))
print('iskeyword("print") :', keyword.iskeyword('print'))

try:
    exec('class = 5')
except SyntaxError as e:
    print('SyntaxError:', e.msg)

class_ = 5
print('а так можно, переменная class_ =', class_)

всего ключевых слов: 35
первые десять : ['False', 'None', 'True', 'and', 'as', 'assert', 'async', 'await', 'break', 'class']
мягкие        : ['_', 'case', 'match']
iskeyword("for")   : True
iskeyword("forest"): False
iskeyword("print") : False
SyntaxError: invalid syntax
а так можно, переменная class_ = 5


### Литералы

<sub>Слайд 37</sub>

* **Литерал** — любая нотация для представления значения прямо в исходном коде.
* Технически: литералу значение присваивается **во время компиляции**, а переменной — **во время выполнения**.

```python
age = 29
nationality = "Hungarian"
```
Здесь `29` и `"Hungarian"` — литералы, `age` и `nationality` — переменные.

* Литералы бывают: числовые (`42`, `3.14`, `1_000_000`, `0b1010`, `0o17`, `0xff`, `2e-3`, `3+4j`), строковые (`'a'`, `"b"`, `"""c"""`, `b'bytes'`, `r'raw'`), а также `True`, `False`, `None`.
* Есть и литералы-конструкторы контейнеров: `[1, 2]`, `(1, 2)`, `{'a': 1}`, `{1, 2}`.
* Литералы попадают в `co_consts` объекта кода, имена — в `co_names`.


**Литералы всех видов**

Напечатаем разные литералы вместе с типами, которые из них получаются,
и заодно посмотрим, что компилятор считает константой, а что нет.


In [ ]:
samples = [42, 1_000_000, 0b1010, 0o17, 0xff, 3.14, 2e-3, 3 + 4j,
           True, None, 'строка', b'bytes', r'C:\новый', (1, 2), [1, 2], {'a': 1}]
for value in samples:
    print(repr(value).ljust(14), '->', type(value).__name__)

print()
rate = 0.5
print('в константах функции с литералом:', (lambda: 100 * 2).__code__.co_consts)
print('в константах функции с именем   :', (lambda: rate * 2).__code__.co_consts)
print('имена той же функции            :', (lambda: rate * 2).__code__.co_names)

42             -> int
1000000        -> int
10             -> int
15             -> int
255            -> int
3.14           -> float
0.002          -> float
(3+4j)         -> complex
True           -> bool
None           -> NoneType
'строка'       -> str
b'bytes'       -> bytes
'C:\\новый'    -> str
(1, 2)         -> tuple
[1, 2]         -> list
{'a': 1}       -> dict

в константах функции с литералом: (None, 200)
в константах функции с именем   : (None, 2)
имена той же функции            : ('rate',)


### Операторы и разделители

<sub>Слайд 38-39</sub>

**Оператор** — символ, используемый для выполнения действия над некоторым значением.

| группа | символы |
|---|---|
| арифметические | `+` `-` `*` `/` `//` `%` `**` |
| сравнения | `<` `<=` `>` `>=` `==` `!=` |
| логические | `and` `or` `not` |
| побитовые | `&` `\|` `^` `~` `<<` `>>` |
| принадлежность и тождество | `in`, `not in`, `is`, `is not` |
| присваивание | `=` и составные `+=` `-=` `*=` `//=` ... |

**Разделитель** — последовательность из одного или нескольких символов, обозначающая
границу между отдельными независимыми областями в тексте или потоке данных.

* Разделители Python: `( ) [ ] { }` `,` `:` `.` `;` `@` `->` и сам **перевод строки**.
* Скобки задают ещё и **неявное продолжение строки**: пока скобка не закрыта, оператор продолжается.
* Явное продолжение — обратный слэш `\` в конце строки; используется редко.


**Операторы и разделители в деле**

Посмотрим на операторы, на скрытый за ними вызов метода
и на то, как скобки позволяют переносить оператор на несколько строк.


In [ ]:
print('деление       :', 7 / 2, '| нацело:', 7 // 2, '| остаток:', 7 % 2)
print('степень       :', 2 ** 10, '| побитовое и:', 5 & 3, '| сдвиг:', 5 << 1)
print('цепочка сравнений:', 2 < 3 <= 4)
print('принадлежность:', 'ра' in 'программа', '| тождество:', None is None)
print('за плюсом стоит метод:', (2).__add__(3), 'ab'.__mul__(3))

total = (10 +
         20 +
         30)
print('неявное продолжение строки:', total)
print('скобка делает кортеж:', type((1, 2)).__name__,
      '| а без запятой это просто число:', type((1)).__name__)

деление       : 3.5 | нацело: 3 | остаток: 1
степень       : 1024 | побитовое и: 1 | сдвиг: 10
цепочка сравнений: True
принадлежность: True | тождество: True
за плюсом стоит метод: 5 ababab
неявное продолжение строки: 60
скобка делает кортеж: tuple | а без запятой это просто число: int


## Переменные, объекты и ссылки


### Всё — объект. Что происходит при `x = 7`

<sub>Слайд 40-41</sub>

* В Python **все данные — объекты**: числа, строки, списки, функции, классы, модули.
* Переменные **создаются в момент присваивания** и до этого не существуют.
* Переменная может ссылаться на объект **любого типа** и должна быть присвоена до того, как к ней обратятся.

Что происходит при выполнении `x = 7`:

1. В памяти создаётся **объект**, представляющий число `7`.
2. Создаётся **переменная** `x`, если её ещё не было.
3. Переменной `x` присваивается **ссылка** на этот объект.

```
   имя            ссылка              объект в памяти
  ┌─────┐                        ┌───────────────────────┐
  │  x  │ ──────────────────────▶│ тип: int  refs: 1  │ 7 │
  └─────┘                        └───────────────────────┘
```

* Внутренне переменная — это **указатель** на область памяти объекта.
* Связь «имя → объект» и называется **ссылкой** (reference): ассоциация, реализованная как указатель.


**Что физически происходит при `x = 7`**

Сначала посмотрим, что бывает при обращении к несуществующему имени,
потом создадим объект и рассмотрим его со всех сторон, а потом трижды переклеим ярлык.


In [ ]:
try:
    print(counter_never_assigned)
except NameError as e:
    print('NameError:', e)

box = 7
print('box =', repr(box), '| тип:', type(box).__name__, '| id:', id(box))
print('размер объекта в байтах:', box.__sizeof__())

box = 'семь'
print('box =', repr(box), '| тип:', type(box).__name__, '| id:', id(box))

box = [7, 7, 7]
print('box =', repr(box), '| тип:', type(box).__name__, '| id:', id(box))
print('шестнадцатеричный адрес:', hex(id(box)))

NameError: name 'counter_never_assigned' is not defined
box = 7 | тип: int | id: 4302124352
размер объекта в байтах: 28
box = 'семь' | тип: str | id: 4380964208
box = [7, 7, 7] | тип: list | id: 5001392768
шестнадцатеричный адрес: 0x12a1b3280


### Кеширование, заголовок объекта, сборка мусора

<sub>Слайд 42</sub>

* В целях оптимизации Python внутренне **кеширует и повторно использует** некоторые неизменяемые объекты: небольшие целые числа и короткие строки. Каждый `0` — не новый участок памяти.
* Каждый объект содержит **два стандартных заголовочных поля**:
  * **обозначение типа** — метка типа объекта;
  * **счётчик ссылок** (`refcount`) — сколько ссылок на объект существует сейчас.
* **Имена не имеют типов; типы обитают в объектах, а не в именах.**
* При присваивании имени нового объекта память предыдущего освобождается, если на него **не ссылается никакое другое имя или объект**.
* Такое автоматическое освобождение памяти называется **сборкой мусора** (garbage collection).
* Каждый раз, когда `x` присваивается новый объект, счётчик ссылок предыдущего уменьшается; при достижении нуля память освобождается.


**Кеш малых целых: 256 и 257**

Классическая демонстрация. Обратите внимание: одно из чисел мы получаем
вычислением во время выполнения, а не литералом — иначе эффект не увидеть.


In [ ]:
import sys

base = 250
a = 256
b = base + 6
print('256:  a == b ->', a == b, '|  a is b ->', a is b)
print('      id(a) =', id(a), ' id(b) =', id(b))

c = 257
d = base + 7
print('257:  c == d ->', c == d, '|  c is d ->', c is d)
print('      id(c) =', id(c), ' id(d) =', id(d))

print()
print('минус пять тоже кеширован:', (-3 - 2) is (base - 255))

256:  a == b -> True |  a is b -> True
      id(a) = 4302132320  id(b) = 4302132320
257:  c == d -> True |  c is d -> False
      id(c) = 5067312112  id(d) = 4988237072

минус пять тоже кеширован: True


<>:15: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:15: SyntaxWarning: "is" with a literal. Did you mean "=="?
/var/folders/z5/mmh4h5s153x_w52pmq7hm3nr0000gn/T/ipykernel_96389/1189262963.py:15: SyntaxWarning: "is" with a literal. Did you mean "=="?
  print('минус пять тоже кеширован:', (-3 - 2) is (base - 255))


**Кеширование строк и `sys.intern`**

То же самое, но со строками: какие строки интерпретатор объединяет сам,
а какие приходится объединять вручную.


In [ ]:
import sys

s1 = 'python'
part = 'pyth'
s2 = part + 'on'
print('python :  ==', s1 == s2, ' is', s1 is s2)

s3 = sys.intern(part + 'on')
print('после intern:  is', s1 is s3)

t1 = 'hello world'
t2 = ' '.join(['hello', 'world'])
print('hello world:  ==', t1 == t2, ' is', t1 is t2)

print('длина s1:', len(s1), '| длина t1:', len(t1))

python :  == True  is False
после intern:  is True
hello world:  == True  is False
длина s1: 6 | длина t1: 11


**`sys.getrefcount` — считаем стрелки**

Посмотрим на счётчик ссылок в действии: как он растёт при появлении новых имён
и падает при их удалении.


In [ ]:
import sys

samples = [10, 20, 30]
print('только что создали:      ', sys.getrefcount(samples))

alias = samples
print('после alias = samples:   ', sys.getrefcount(samples))

holder = [samples, samples]
print('после двух ссылок в списке:', sys.getrefcount(samples))

del alias
print('после del alias:         ', sys.getrefcount(samples))

del holder
print('после del holder:        ', sys.getrefcount(samples))

только что создали:       2
после alias = samples:    3
после двух ссылок в списке: 5
после del alias:          4
после del holder:         2


### Разделяемые ссылки

<sub>Слайд 43-44</sub>

```python
x = 7
y = x      # два имени — один объект
x = 3      # x перевесили, y не тронули
```

* Ситуация, когда **несколько имён ссылаются на один и тот же объект**, называется **разделяемой ссылкой** (shared reference), а сам объект — разделяемым.
* В отличие от ряда языков, переменные Python — **всегда указатели на объекты**, а не метки изменяемой памяти.
* Установка переменной в новое значение **не изменяет прежний объект**, а заставляет переменную ссылаться на совершенно другой объект.
* Поэтому после `x = 3` переменная `y` по-прежнему равна `7`.
* Счётчик ссылок объекта можно узнать: `sys.getrefcount(x)` (модуль `sys`).


**`x = 7; y = x; x = 3` — следим за адресами**

Повторим слайд построчно, печатая после каждого шага и значения, и адреса.


In [ ]:
x = 7
y = x
print('после y = x   : x =', x, 'y =', y)
print('               id(x) =', id(x), 'id(y) =', id(y))
print('               x is y ->', x is y)

x = 3
print('после x = 3   : x =', x, 'y =', y)
print('               id(x) =', id(x), 'id(y) =', id(y))
print('               x is y ->', x is y)

print()
print('объект 7 никуда не делся — на него смотрит y')

после y = x   : x = 7 y = 7
               id(x) = 4302124352 id(y) = 4302124352
               x is y -> True
после x = 3   : x = 3 y = 7
               id(x) = 4302124224 id(y) = 4302124352
               x is y -> False

объект 7 никуда не делся — на него смотрит y


**А теперь то же самое со списком**

Точно та же конструкция из двух имён, но объект — изменяемый список.
Вот здесь поведение внезапно другое.


In [ ]:
numbers = [1, 2, 3]
same_numbers = numbers
print('оба имени:', numbers, same_numbers, '| is:', numbers is same_numbers)

same_numbers.append(99)
print('после same_numbers.append(99)')
print('  numbers      =', numbers)
print('  same_numbers =', same_numbers)
print('  id совпадают :', id(numbers) == id(same_numbers))

same_numbers = [0, 0]
print('после same_numbers = [0, 0]')
print('  numbers      =', numbers)
print('  same_numbers =', same_numbers)

оба имени: [1, 2, 3] [1, 2, 3] | is: True
после same_numbers.append(99)
  numbers      = [1, 2, 3, 99]
  same_numbers = [1, 2, 3, 99]
  id совпадают : True
после same_numbers = [0, 0]
  numbers      = [1, 2, 3, 99]
  same_numbers = [0, 0]


**Копии: срез, `copy.copy` и `copy.deepcopy`**

Раз присваивание не копирует, посмотрим, что копирует на самом деле —
и чем поверхностная копия отличается от глубокой.


In [ ]:
import copy

original = [[1, 2], [3, 4]]
by_slice = original[:]
by_copy = copy.copy(original)
by_deep = copy.deepcopy(original)

print('верхний уровень — разные объекты:')
print('  original is by_slice:', original is by_slice)
print('внутренние списки:')
print('  by_slice[0] is original[0]:', by_slice[0] is original[0])
print('  by_deep[0]  is original[0]:', by_deep[0] is original[0])

original[0].append('!!!')
print('после original[0].append:')
print('  by_slice =', by_slice, ' <- испортился')
print('  by_deep  =', by_deep, ' <- цел')

верхний уровень — разные объекты:
  original is by_slice: False
внутренние списки:
  by_slice[0] is original[0]: True
  by_deep[0]  is original[0]: False
после original[0].append:
  by_slice = [[1, 2, '!!!'], [3, 4]]  <- испортился
  by_deep  = [[1, 2], [3, 4]]  <- цел


### Проверка на равенство: `==` против `is`

<sub>Слайд 45</sub>

* Операция `==` проверяет, **имеют ли два объекта одинаковые значения**. Это почти всегда то, что вам нужно.
* Операция `is` проверяет **идентичность**: истинна только если оба имени указывают на **точно тот же самый объект**. Это гораздо более строгая проверка.
* По сути `is` просто **сравнивает указатели**, реализующие ссылки, и служит способом обнаружить разделяемые ссылки в коде.

```python
L1 = [1, 2, 3]
L2 = [1, 2, 3]
L1 == L2     # True   — значения равны
L1 is L2     # False  — объекты разные

x = 7
y = 7
x == y       # True
x is y       # True   — сработал кеш малых целых!
```

* `is` в реальном коде применяют почти исключительно так: `x is None`, `x is not None`.


**`L1 == L2` истинно, `L1 is L2` ложно**

Слайд буквально: два списка с одинаковым содержимым и одно число,
которому случайно повезло попасть в кеш.


In [ ]:
L1 = [1, 2, 3]
L2 = [1, 2, 3]
print('L1 == L2:', L1 == L2, '| L1 is L2:', L1 is L2)
print('id:', id(L1), id(L2))

L3 = L1
print('L1 is L3:', L1 is L3, '(вот это и есть разделяемая ссылка)')

x = 7
y = 7
print('x == y:', x == y, '| x is y:', x is y, '(случайность кеша)')

big1 = 10 ** 6
big2 = 10 ** 3 * 10 ** 3
print('big1 == big2:', big1 == big2, '| big1 is big2:', big1 is big2)

L1 == L2: True | L1 is L2: False
id: 4380695232 4380941696
L1 is L3: True (вот это и есть разделяемая ссылка)
x == y: True | x is y: True (случайность кеша)
big1 == big2: True | big1 is big2: False


**Ловушка: `x is None` против `x == None`**

Почему проверку на `None` всегда пишут через `is`.
Заведём класс, чей `==` «соглашается» со всем подряд.


In [ ]:
class AlwaysEqual:
    def __eq__(self, other):
        return True
    def __repr__(self):
        return '<AlwaysEqual>'

value = AlwaysEqual()
print('value == None :', value == None)
print('value is None :', value is None)

nothing = None
print('nothing is None:', nothing is None)
print('None — синглтон, объект в программе ровно один:')
print('  id(None) везде одинаков:', id(None) == id(nothing))

value == None : True
value is None : False
nothing is None: True
None — синглтон, объект в программе ровно один:
  id(None) везде одинаков: True


**Изменяемый объект в аргументе по умолчанию**

Знаменитая ловушка. Подробно разберём её на третьем занятии,
но модель памяти, которая её объясняет, у вас уже есть.


In [ ]:
def collect(item, basket=[]):
    basket.append(item)
    return basket

print('первый вызов :', collect('яблоко'))
print('второй вызов :', collect('груша'))
print('третий вызов :', collect('слива'))

print('объект по умолчанию живёт в самой функции:')
print('  collect.__defaults__ =', collect.__defaults__)

def collect_ok(item, basket=None):
    if basket is None:
        basket = []
    basket.append(item)
    return basket

print('правильный вариант:', collect_ok('яблоко'), collect_ok('груша'))

первый вызов : ['яблоко']
второй вызов : ['яблоко', 'груша']
третий вызов : ['яблоко', 'груша', 'слива']
объект по умолчанию живёт в самой функции:
  collect.__defaults__ = (['яблоко', 'груша', 'слива'],)
правильный вариант: ['яблоко'] ['груша']


## Числовые типы


### `int` и `float`

<sub>Слайд 46-47</sub>

**`int` — целые числа**

* Имеют **неограниченную точность**: могут содержать столько цифр, сколько помещается в доступной памяти.
* Нет `int8/int32/int64`, нет переполнения, нет `MAX_INT`.
* Плата — скорость и память: большое число хранится как массив «цифр» по основанию 2³⁰.

**`float` — числа с плавающей точкой**

* В стандартном интерпретаторе CPython реализованы как `double` языка C.
* Точность — та, что даёт компилятор C, которым собран интерпретатор: двойная точность IEEE 754, 64 бита, ≈ 15–17 значащих десятичных цифр.
* Отсюда все привычные неточности: `0.1 + 0.2 != 0.3`.
* Характеристики доступны в `sys.float_info`.


**Целые без переполнения: `2**1000` и `100!`**

Посчитаем два заведомо огромных числа и посмотрим,
сколько в них десятичных цифр и сколько байт они занимают.


In [ ]:
import math, sys

power = 2 ** 1000
fact = math.factorial(100)

print('2**1000 начинается с:', str(power)[:30], '...')
print('  цифр в записи :', len(str(power)))
print('  байт в памяти :', sys.getsizeof(power))

print('100! заканчивается на:', '...' + str(fact)[-30:])
print('  цифр в записи :', len(str(fact)))
print('  нулей на конце:', len(str(fact)) - len(str(fact).rstrip('0')))

print('точное деление больших целых:', (fact * 7) // 7 == fact)

2**1000 начинается с: 107150860718626732094842504906 ...
  цифр в записи : 302
  байт в памяти : 160
100! заканчивается на: ...916864000000000000000000000000
  цифр в записи : 158
  нулей на конце: 24
точное деление больших целых: True


**`0.1 + 0.2` и где именно теряется точность**

Самая известная особенность вещественной арифметики.
Посмотрим не только на результат, но и на то, чем на самом деле является `0.1`.


In [ ]:
print('0.1 + 0.2       =', 0.1 + 0.2)
print('0.1 + 0.2 == 0.3:', 0.1 + 0.2 == 0.3)
print('разница         :', 0.1 + 0.2 - 0.3)

print()
print('что на самом деле лежит в 0.1:')
print(' ', '%.20f' % 0.1)
print('точное значение как дробь:', (0.1).as_integer_ratio())

print()
print('со слайда: 0.1+0.1+0.1-0.3 =', 0.1 + 0.1 + 0.1 - 0.3)
print('а вот 0.5 и 0.25 представимы точно:', 0.5 + 0.25 == 0.75)

0.1 + 0.2       = 0.30000000000000004
0.1 + 0.2 == 0.3: False
разница         : 5.551115123125783e-17

что на самом деле лежит в 0.1:
  0.10000000000000000555
точное значение как дробь: (3602879701896397, 36028797018963968)

со слайда: 0.1+0.1+0.1-0.3 = 5.551115123125783e-17
а вот 0.5 и 0.25 представимы точно: True


**`sys.float_info`, бесконечность и `nan`**

Посмотрим на паспорт типа `float` и на три особых значения,
которые определены стандартом IEEE 754.


In [ ]:
import sys, math

info = sys.float_info
print('макс. float :', info.max)
print('мин. положительный:', info.min)
print('надёжных десятичных цифр:', info.dig)
print('машинный эпсилон:', info.epsilon)

plus_inf = float('inf')
not_a_number = float('nan')
print()
print('inf:', plus_inf, '| inf + 1 =', plus_inf + 1)
print('1e308 * 10 =', 1e308 * 10, '(переполнение -> inf)')
print('nan:', not_a_number)
print('nan == nan ->', not_a_number == not_a_number)
print('math.isnan(nan) ->', math.isnan(not_a_number))

макс. float : 1.7976931348623157e+308
мин. положительный: 2.2250738585072014e-308
надёжных десятичных цифр: 15
машинный эпсилон: 2.220446049250313e-16

inf: inf | inf + 1 = inf
1e308 * 10 = inf (переполнение -> inf)
nan: nan
nan == nan -> False
math.isnan(nan) -> True


**Правильное сравнение: `math.isclose`**

Раз `==` для вещественных не годится, посмотрим,
чем его заменяют в реальном коде.


In [ ]:
import math

a = 0.1 + 0.2
b = 0.3
print('a == b        :', a == b)
print('isclose(a, b) :', math.isclose(a, b))
print('вручную |a-b| < 1e-9:', abs(a - b) < 1e-9)

print()
print('относительный допуск важен на больших числах:')
big_a, big_b = 1e16, 1e16 + 1.0
print('  1e16 и 1e16+1 равны как float:', big_a == big_b)
print('  isclose:', math.isclose(big_a, big_b))

print('около нуля нужен абсолютный допуск:')
print('  isclose(0.0, 1e-20):', math.isclose(0.0, 1e-20))
print('  с abs_tol=1e-12    :', math.isclose(0.0, 1e-20, abs_tol=1e-12))

a == b        : False
isclose(a, b) : True
вручную |a-b| < 1e-9: True

относительный допуск важен на больших числах:
  1e16 и 1e16+1 равны как float: True
  isclose: True
около нуля нужен абсолютный допуск:
  isclose(0.0, 1e-20): False
  с abs_tol=1e-12    : True


### Комплексные числа

<sub>Слайд 48</sub>

* Литерал комплексного числа: `действительная_часть + мнимая_часть`, где мнимая часть заканчивается символом `j` или `J`.
* Формально действительная часть **необязательна**, поэтому мнимая может стоять сама по себе: `3j`.
* Внутренне комплексное число — **пара чисел с плавающей точкой** (`.real` и `.imag`, оба `float`).
* Все числовые операции над такими числами выполняют **математику комплексных чисел**.
* Полезное: `abs(z)` — модуль, `z.conjugate()` — сопряжённое, модуль `cmath` — комплексные версии `sqrt`, `exp`, `log`.


**Комплексные числа в деле**

Создадим комплексное число двумя способами, посмотрим на его части
и проверим главное равенство математики.


In [ ]:
import cmath

z = 3 + 4j
w = complex(1, -1)
print('z =', z, '| w =', w)
print('z.real =', z.real, '| z.imag =', z.imag, '| типы:', type(z.real).__name__)
print('|z| =', abs(z), '| сопряжённое:', z.conjugate())
print('z * w =', z * w, '| z / w =', z / w)

print()
print('корень из -1:', cmath.sqrt(-1))
euler = cmath.exp(1j * cmath.pi) + 1
print('exp(i*pi) + 1 =', euler, '(ноль с точностью до float)')

try:
    print(z < w)
except TypeError as e:
    print('TypeError:', e)

z = (3+4j) | w = (1-1j)
z.real = 3.0 | z.imag = 4.0 | типы: float
|z| = 5.0 | сопряжённое: (3-4j)
z * w = (7+1j) | z / w = (-0.5+3.5j)

корень из -1: 1j
exp(i*pi) + 1 = 1.2246467991473532e-16j (ноль с точностью до float)
TypeError: '<' not supported between instances of 'complex' and 'complex'


### `bool` — и почему он наследуется от `int`

<sub>Слайд 49</sub>

* Тип принимает два значения: `True` и `False`.
* Класс `bool` **наследуется от класса `int`**: `issubclass(bool, int)` → `True`.
* Объекты `True` и `False` ведут себя как целые `1` и `0`, за исключением вывода: у них переопределены `str` и `repr`, поэтому печатаются словами, а не цифрами.

```python
True + 4        # 5
True == 1       # True
isinstance(True, int)   # True
sum([True, True, False])  # 2
```

* `True` и `False` — синглтоны: объект каждого ровно один.
* `bool(x)` работает для любого объекта: пусто или ноль → `False`, всё остальное → `True`.


**`True + 4` и подсчёт через `sum`**

Проверим родство `bool` и `int` и сразу посмотрим,
как из него получается самая частая идиома подсчёта.


In [ ]:
print('True + 4        =', True + 4)
print('True * 10       =', True * 10)
print('False - 1       =', False - 1)
print('issubclass(bool, int):', issubclass(bool, int))
print('isinstance(True, int):', isinstance(True, int))

flags = [True, True, False, True, False]
print('sum(flags) =', sum(flags), 'из', len(flags))

scores = [0.9, 0.4, 0.7, 0.2, 0.95]
above = [s > 0.5 for s in scores]
print('признаки выше порога:', above)
print('сколько прошло порог:', sum(above))
print('доля:', sum(above) / len(scores))

True + 4        = 5
True * 10       = 10
False - 1       = -1
issubclass(bool, int): True
isinstance(True, int): True
sum(flags) = 3 из 5
признаки выше порога: [True, False, True, False, True]
сколько прошло порог: 3
доля: 0.6


### Рациональные числа: `fractions.Fraction`

<sub>Слайд 50</sub>

* `Fraction` — тип для **рациональных чисел**: явно хранит **числитель и знаменатель**.
* За счёт этого избегает неточностей и ограничений плавающей арифметики.
* Дробь автоматически сокращается: `Fraction(6, 8)` → `Fraction(3, 4)`.

```python
0.1 + 0.1 + 0.1 - 0.3  # 5.551115123125783e-17

Fraction(1,10) + Fraction(1,10) + Fraction(1,10) - Fraction(3,10)
# Fraction(0, 1)  — точный ноль
```

* Создание: `Fraction(1, 3)`, `Fraction('0.25')`, `Fraction(0.25)` (из `float` — точно, со всей его погрешностью).
* Цена: операции медленнее, знаменатели растут.


**`Fraction`: точная арифметика**

Повторим пример со слайда и посмотрим,
чем отличается создание дроби из строки и из `float`.


In [ ]:
from fractions import Fraction

tenth = Fraction(1, 10)
print('сумма трёх десятых минус три десятых:',
      tenth + tenth + tenth - Fraction(3, 10))
print('float даёт:', 0.1 + 0.1 + 0.1 - 0.3)

print()
print('Fraction(6, 8) сокращается ->', Fraction(6, 8))
print('из строки  :', Fraction('0.1'))
print('из float   :', Fraction(0.1))
print('одна треть :', Fraction(1, 3), '=', float(Fraction(1, 3)))
print('точная сумма 1/3 + 1/6:', Fraction(1, 3) + Fraction(1, 6))

сумма трёх десятых минус три десятых: 0
float даёт: 5.551115123125783e-17

Fraction(6, 8) сокращается -> 3/4
из строки  : 1/10
из float   : 3602879701896397/36028797018963968
одна треть : 1/3 = 0.3333333333333333
точная сумма 1/3 + 1/6: 1/2


### Десятичные числа: `decimal.Decimal`

<sub>Слайд 51-52</sub>

* `decimal` — числа с плавающей точкой **фиксированной точности**: функционально похожи на `float`, но имеют фиксированное количество разрядов после десятичной точки.
* Хранятся в **десятичной** системе, поэтому `Decimal('0.1')` — это ровно одна десятая.
* В общем случае **работают медленнее** обычного `float`.
* Хорошо подходят для величин с фиксированной точностью — прежде всего **денежных сумм** — и помогают достичь лучшей числовой точности.
* Контекст `decimal.getcontext()` задаёт точность (`prec`) и правило округления (`rounding`).
* **Создавать только из строки**: `Decimal('0.1')`, а не `Decimal(0.1)`.


**`Decimal` и денежные расчёты**

Сравним `float` и `Decimal` на счёте с копейками
и посмотрим, как задаётся точность и округление.


In [ ]:
from decimal import Decimal, getcontext, ROUND_HALF_UP

print('float   : 0.1 * 3 =', 0.1 * 3)
print('Decimal : ', Decimal('0.1') * 3)
print('из float — видно правду:', Decimal(0.1))

prices = ['19.99', '5.49', '3.50', '0.99']
total_float = sum(float(p) for p in prices)
total_dec = sum(Decimal(p) for p in prices)
print()
print('итого float  :', repr(total_float))
print('итого Decimal:', total_dec)

with_tax = total_dec * Decimal('1.20')
print('с НДС 20%    :', with_tax)
print('округлили до копеек:',
      with_tax.quantize(Decimal('0.01'), rounding=ROUND_HALF_UP))
print('точность контекста:', getcontext().prec, 'значащих цифр')

float   : 0.1 * 3 = 0.30000000000000004
Decimal :  0.3
из float — видно правду: 0.1000000000000000055511151231257827021181583404541015625

итого float  : 29.969999999999995
итого Decimal: 29.97
с НДС 20%    : 35.9640
округлили до копеек: 35.96
точность контекста: 28 значащих цифр


## Создание чисел и арифметика


### Как создать число

<sub>Слайд 53</sub>

**Числовым литералом:**

| Тип | Примеры |
|---|---|
| `int` | `3`, `100_200_3000`, `0o177` (8-ричное), `0x9ff` (16-ричное), `0b101010` (2-ичное) |
| `float` | `1.37`, `1.`, `3.14e-10` |
| `complex` | `3+4j`, `3j` |

**Встроенной функцией:**

* `int()`, `oct()`, `hex()`, `bin()`
* `float()`
* `complex()`
* `bool()`
* `fractions.Fraction()`
* `decimal.Decimal()`

**Оператором:** результат любого арифметического выражения — тоже число.


**Литералы всех видов**

Напечатаем все формы записи со слайда и убедимся,
что число — это число, независимо от того, как оно записано.


In [ ]:
big = 100_200_3000
print('100_200_3000 ->', big, '| тип:', type(big).__name__)

print('0o177      ->', 0o177)
print('0x9ff      ->', 0x9ff)
print('0b101010   ->', 0b101010)
print('0xFF == 255:', 0xFF == 255)

print()
print('1.37       ->', 1.37)
print('1.         ->', 1., '| тип:', type(1.).__name__)
print('3.14e-10   ->', 3.14e-10)
print('3+4j       ->', 3 + 4j, '| 3j ->', 3j)
print('1_000_000 == 1000000:', 1_000_000 == 1000000)

100_200_3000 -> 1002003000 | тип: int
0o177      -> 127
0x9ff      -> 2559
0b101010   -> 42
0xFF == 255: True

1.37       -> 1.37
1.         -> 1.0 | тип: float
3.14e-10   -> 3.14e-10
3+4j       -> (3+4j) | 3j -> 3j
1_000_000 == 1000000: True


**`bin`, `hex`, `oct` и битовые операции**

Обратное преобразование — число в строку — и операции над битами.


In [ ]:
value = 42
print('bin:', bin(value), '| hex:', hex(value), '| oct:', oct(value))
print('тип bin(42):', type(bin(value)).__name__, '<- это строка!')
print('обратно из строки:', int('101010', 2), int('2a', 16))

a, b = 0b1100, 0b1010
print()
print('a =', bin(a), ' b =', bin(b))
print('a & b  (И)      =', bin(a & b))
print('a | b  (ИЛИ)    =', bin(a | b))
print('a ^ b  (XOR)    =', bin(a ^ b))
print('a << 2 (сдвиг)  =', bin(a << 2), '=', a << 2)
print('a >> 2          =', bin(a >> 2), '=', a >> 2)
print('количество единиц в 42:', bin(value).count('1'))

bin: 0b101010 | hex: 0x2a | oct: 0o52
тип bin(42): str <- это строка!
обратно из строки: 42 42

a = 0b1100  b = 0b1010
a & b  (И)      = 0b1000
a | b  (ИЛИ)    = 0b1110
a ^ b  (XOR)    = 0b110
a << 2 (сдвиг)  = 0b110000 = 48
a >> 2          = 0b11 = 3
количество единиц в 42: 3


### Смешанные выражения: приведение к «самому сложному»

<sub>Слайд 54</sub>

* В выражениях с **разнородными** числовыми типами Python сначала преобразует операнды к типу **самого сложного** операнда, затем выполняет операцию над операндами одного типа.
* Иерархия сложности:

```
bool  →  int  →  float  →  complex
```

* Целые проще чисел с плавающей точкой, те — проще комплексных.
* `30 + 3.7` → целое преобразуется в `float`, результат `33.7` — тоже `float`.
* `Decimal` и `Fraction` в эту цепочку встроены частично: `Decimal` **не смешивается** с `float` (будет `TypeError`), `Fraction` с `float` — смешивается, давая `float`.


**Смешанная арифметика и приоритет операций**

Проверим иерархию типов и заодно разберём три места,
где приоритет операций даёт неожиданный результат.


In [ ]:
from decimal import Decimal
from fractions import Fraction

print('30 + 3.7      =', 30 + 3.7, '|', type(30 + 3.7).__name__)
print('True + 1      =', True + 1, '|', type(True + 1).__name__)
print('2 + 3.0 + 1j  =', 2 + 3.0 + 1j, '|', type(2 + 3.0 + 1j).__name__)
print('Fraction+float=', Fraction(1, 2) + 0.5, '|', type(Fraction(1, 2) + 0.5).__name__)

try:
    print(Decimal('0.1') + 0.1)
except TypeError as e:
    print('TypeError:', e)

print()
print('2 ** 3 ** 2 =', 2 ** 3 ** 2, '(степень правоассоциативна: 2**9)')
print('-2 ** 2     =', -2 ** 2, '(унарный минус слабее степени)')
print('(-2) ** 2   =', (-2) ** 2)
print('2 + 3 * 4   =', 2 + 3 * 4)

30 + 3.7      = 33.7 | float
True + 1      = 2 | int
2 + 3.0 + 1j  = (5+1j) | complex
Fraction+float= 1.0 | float
TypeError: unsupported operand type(s) for +: 'decimal.Decimal' and 'float'

2 ** 3 ** 2 = 512 (степень правоассоциативна: 2**9)
-2 ** 2     = -4 (унарный минус слабее степени)
(-2) ** 2   = 4
2 + 3 * 4   = 14


### Деление: `/`, `//` и `%`

<sub>Слайд 55</sub>

* Операция `/` всегда выполняет **настоящее деление**, возвращая `float` с остатком — **независимо от типов операндов**.
* Операция `//` выполняет **деление с округлением в меньшую сторону** (floor division): усекает остаток и возвращает `int` для целых операндов или `float`, если хотя бы один операнд вещественный.

```python
10 / 4      # 2.5
10 // 4     # 2
10 // 4.0   # 2.0
```

* `%` — остаток, согласованный с `//`: всегда выполняется `a == (a // b) * b + a % b`.
* **Округление именно вниз**, а не к нулю: `-7 // 2 == -4`, а не `-3`. Знак остатка совпадает со знаком **делителя**: `-7 % 2 == 1`.
* `divmod(a, b)` возвращает пару `(a // b, a % b)` за одно вычисление.


**Деление с отрицательными числами**

Самое важное место слайда: сравним `//` с округлением к нулю
и посмотрим, как ведёт себя остаток.


In [ ]:
import math

print(' 10 / 4   =', 10 / 4, '  |  10 // 4 =', 10 // 4, ' | 10 // 4.0 =', 10 // 4.0)
print(' 10 % 4   =', 10 % 4, '   | divmod(10, 4) =', divmod(10, 4))

print()
print('-7 // 2   =', -7 // 2, '  <- округление ВНИЗ, как в математике')
print('int(-7/2) =', int(-7 / 2), '  <- усечение К НУЛЮ, как в C и Java')
print('math.trunc(-7/2) =', math.trunc(-7 / 2))
print('-7 % 2    =', -7 % 2, '   <- знак остатка = знак делителя')
print(' 7 % -2   =', 7 % -2)
print('divmod(-7, 2) =', divmod(-7, 2))
print('тождество a == (a//b)*b + a%b :', (-7 // 2) * 2 + (-7 % 2) == -7)

 10 / 4   = 2.5   |  10 // 4 = 2  | 10 // 4.0 = 2.0
 10 % 4   = 2    | divmod(10, 4) = (2, 2)

-7 // 2   = -4   <- округление ВНИЗ, как в математике
int(-7/2) = -3   <- усечение К НУЛЮ, как в C и Java
math.trunc(-7/2) = -3
-7 % 2    = 1    <- знак остатка = знак делителя
 7 % -2   = -1
divmod(-7, 2) = (-4, 1)
тождество a == (a//b)*b + a%b : True
